# Lab 26 — Fixed Hungarian external-test baseline

Notebook chính thức cho protocol đã chốt:

- **Test cố định:** Hungarian.
- **Development:** Cleveland, Switzerland, VA.
- **Inner CV:** 3 folds theo hospital; mỗi fold dùng 2 hospital train + 1 hospital validation.
- **Final fit:** train trên toàn bộ 3 development hospitals.
- **Final test:** Hungarian đúng một lần sau khi mọi quyết định đã được khóa.

Baseline giữ nguyên tinh thần Colab 15:

- Logistic Regression và LightGBM.
- P1: chỉ chuyển sentinel \`?\` và các giá trị \`trestbps/chol <= 0\` thành missing.
- Train-fold median/mode imputation + missing indicators + one-hot.
- StandardScaler cho Logistic Regression.
- \`class_weight='balanced'\`.
- Threshold cố định \`0.50\`.
- Không Optuna, không SMOTE, không synthetic data, không stacking.

> Lưu ý phương pháp: kết quả LOCO của Colab 24/25 đã được xem, nên kết quả Hungarian ở notebook này không được gọi là blind tuyệt đối. Đây là **fixed external holdout evaluation**. Blind tuyệt đối cần thêm hospital/dataset thứ 5.

In [ ]:
!pip -q install lightgbm seaborn

In [ ]:
import json
import random
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

RANDOM_STATE = 42
FINAL_SEED = 42
INNER_SEEDS = (42, 123, 2025)
THRESHOLD = 0.50

OUTPUT_DIR = Path('/content/uci_multicenter_fixed_hungarian_baseline_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal'
]
TARGET = 'target'
NUMERICAL_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CATEGORICAL_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {
    'cleveland': 'processed.cleveland.data',
    'hungarian': 'processed.hungarian.data',
    'switzerland': 'processed.switzerland.data',
    'va': 'processed.va.data',
}
EXPECTED_ROWS = {'cleveland': 303, 'hungarian': 294, 'switzerland': 123, 'va': 200}
DEVELOPMENT_SITES = ['cleveland', 'switzerland', 'va']
FINAL_TEST_SITE = 'hungarian'
MODEL_NAMES = ['Logistic Regression', 'LightGBM']
COLUMNS = FEATURES + ['num']

LOCAL_DATA_DIR_CANDIDATES = [
    Path('/content/heart-disease-diagnosis/data/raw/uci_multicenter'),
    Path('/content/data/raw/uci_multicenter'),
    Path('data/raw/uci_multicenter'),
    Path('../data/raw/uci_multicenter'),
]
LOCAL_DATA_DIR = next(
    (path for path in LOCAL_DATA_DIR_CANDIDATES if path.exists()),
    None,
)

print('Output directory:', OUTPUT_DIR)
print('Development sites:', DEVELOPMENT_SITES)
print('Final test site:', FINAL_TEST_SITE)
print('Inner seeds:', INNER_SEEDS)
print('Final seed:', FINAL_SEED)

In [ ]:
def read_uci(site, filename):
    source = (LOCAL_DATA_DIR / filename) if LOCAL_DATA_DIR else f'{BASE_URL}/{filename}'
    frame = pd.read_csv(
        source,
        names=COLUMNS,
        na_values=['?'],
        skipinitialspace=True,
    )
    frame = frame.apply(pd.to_numeric, errors='coerce')
    frame[TARGET] = (frame['num'] > 0).astype('int8')
    frame['site'] = site
    return frame[FEATURES + [TARGET, 'site']]

data = pd.concat(
    [read_uci(site, filename) for site, filename in FILES.items()],
    ignore_index=True,
)

assert len(data) == 920, f'Expected 920 rows, got {len(data)}'
for site, expected_rows in EXPECTED_ROWS.items():
    actual_rows = int((data['site'] == site).sum())
    assert actual_rows == expected_rows, (
        f'{site}: expected {expected_rows}, got {actual_rows}'
    )

development = data[data['site'].isin(DEVELOPMENT_SITES)].reset_index(drop=True)
final_test = data[data['site'] == FINAL_TEST_SITE].reset_index(drop=True)

assert len(development) == 626
assert len(final_test) == 294
assert not set(development['site']).intersection({FINAL_TEST_SITE})
assert set(development['site']) == set(DEVELOPMENT_SITES)

# The Hungarian labels are kept out of all development summaries and CV.
development_summary = development.groupby('site').agg(
    rows=(TARGET, 'size'),
    disease_count=(TARGET, 'sum'),
    positive_rate=(TARGET, 'mean'),
).reset_index()
development_missing_rate = development.groupby('site')[FEATURES].apply(
    lambda frame: float(frame.isna().mean().mean())
).rename('missing_rate').reset_index()
development_summary = development_summary.merge(
    development_missing_rate,
    on='site',
    validate='one_to_one',
)

print('Data source:', str(LOCAL_DATA_DIR) if LOCAL_DATA_DIR else 'UCI URL fallback')
print('Development shape:', development.shape)
print('Reserved Hungarian rows:', len(final_test), '(labels not used before final test)')
display(development_summary.round(4))
display(pd.crosstab(development['site'], development[TARGET], margins=True))

development_missing = development.groupby('site')[FEATURES].apply(
    lambda frame: frame.isna().mean()
).T
display((development_missing * 100).round(1).rename(columns=lambda name: f'{name} missing %'))

development_summary.to_csv(OUTPUT_DIR / 'development_site_summary.csv', index=False)
development_missing.to_csv(OUTPUT_DIR / 'development_missing_by_site.csv')
(final_test[['site']].assign(rows=1).groupby('site').sum()).to_csv(
    OUTPUT_DIR / 'final_test_manifest.csv'
)

In [ ]:
def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors='coerce')

    # P1 rule retained from the old baseline:
    # non-positive trestbps/chol are treated as implausible missing values.
    for column in ['trestbps', 'chol']:
        out.loc[out[column] <= 0, column] = np.nan

    return out


def make_preprocessor():
    numerical = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
    ])
    categorical = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ])
    return ColumnTransformer([
        ('numerical', numerical, NUMERICAL_FEATURES),
        ('categorical', categorical, CATEGORICAL_FEATURES),
    ])


def make_models(seed):
    return {
        'Logistic Regression': LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            random_state=seed,
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=250,
            learning_rate=0.03,
            num_leaves=15,
            min_child_samples=15,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            class_weight='balanced',
            random_state=seed,
            verbosity=-1,
            n_jobs=1,
        ),
    }


def make_pipeline(model_name, seed):
    return Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', make_models(seed)[model_name]),
    ])


def score_probability(y_true, probability):
    prediction = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, prediction, labels=[0, 1]
    ).ravel()
    return {
        'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'pr_auc': average_precision_score(y_true, probability),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': (
            roc_auc_score(y_true, probability)
            if len(np.unique(y_true)) > 1 else np.nan
        ),
        'brier': brier_score_loss(y_true, probability),
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_positives': int(tp),
    }


def metric_record(
    evaluation,
    model_name,
    validation_site,
    seed,
    y_true,
    probability,
    fit_seconds,
):
    return {
        'evaluation': evaluation,
        'model': model_name,
        'validation_site': validation_site,
        'seed': seed,
        'n_rows': int(len(y_true)),
        'threshold': THRESHOLD,
        'fit_seconds': fit_seconds,
        **score_probability(y_true, probability),
    }

## Inner validation — development hospitals only

Inner CV là kiểm tra phát triển, không phải external test:

- Fold 1: train Cleveland + VA, validate Switzerland.
- Fold 2: train Cleveland + Switzerland, validate VA.
- Fold 3: train Switzerland + VA, validate Cleveland.

Mỗi seed tạo lại pipeline và model từ đầu. Hungarian không xuất hiện trong bất kỳ fold nào.

In [ ]:
from sklearn.model_selection import GroupKFold

inner_records = []
oof_by_seed = {}

for seed in INNER_SEEDS:
    oof_by_seed[seed] = {
        model_name: np.full(len(development), np.nan, dtype='float64')
        for model_name in MODEL_NAMES
    }

    groups = development['site'].to_numpy()
    labels = development[TARGET].to_numpy()
    splitter = GroupKFold(n_splits=3)

    for fold_number, (fit_idx, valid_idx) in enumerate(
        splitter.split(development, labels, groups),
        start=1,
    ):
        fit_frame = development.iloc[fit_idx].reset_index(drop=True)
        valid_frame = development.iloc[valid_idx].reset_index(drop=True)
        validation_site = valid_frame['site'].iloc[0]

        assert valid_frame['site'].nunique() == 1
        assert not set(fit_frame['site']).intersection({FINAL_TEST_SITE})
        assert not set(valid_frame['site']).intersection({FINAL_TEST_SITE})

        print(
            f'Seed {seed} | fold {fold_number} | '
            f'train={sorted(fit_frame["site"].unique())} | '
            f'valid={validation_site}'
        )

        for model_name in MODEL_NAMES:
            pipeline = make_pipeline(model_name, seed)
            started = time.perf_counter()
            pipeline.fit(
                fit_frame[FEATURES],
                fit_frame[TARGET],
            )
            fit_seconds = time.perf_counter() - started
            probability = pipeline.predict_proba(
                valid_frame[FEATURES]
            )[:, 1]

            oof_by_seed[seed][model_name][valid_idx] = probability
            inner_records.append(metric_record(
                evaluation='inner_cv',
                model_name=model_name,
                validation_site=validation_site,
                seed=seed,
                y_true=valid_frame[TARGET].to_numpy(),
                probability=probability,
                fit_seconds=fit_seconds,
            ))

inner_cv_results = pd.DataFrame(inner_records)
assert len(inner_cv_results) == len(INNER_SEEDS) * 3 * len(MODEL_NAMES)
display(inner_cv_results.round(4))

In [ ]:
inner_site_summary = inner_cv_results.groupby(
    ['model', 'validation_site']
).agg(
    roc_auc=('roc_auc', 'mean'),
    pr_auc=('pr_auc', 'mean'),
    recall=('recall', 'mean'),
    specificity=('specificity', 'mean'),
    f1=('f1', 'mean'),
    brier=('brier', 'mean'),
    false_negatives=('false_negatives', 'mean'),
).reset_index()

inner_worst_site = inner_site_summary.groupby('model').agg(
    roc_auc_worst=('roc_auc', 'min'),
    pr_auc_worst=('pr_auc', 'min'),
    recall_worst=('recall', 'min'),
    brier_worst=('brier', 'max'),
).reset_index()

pooled_inner_records = []
for seed in INNER_SEEDS:
    for model_name in MODEL_NAMES:
        probability = oof_by_seed[seed][model_name]
        assert not np.isnan(probability).any()
        pooled_inner_records.append({
            'model': model_name,
            'seed': seed,
            **score_probability(
                development[TARGET].to_numpy(),
                probability,
            ),
        })

pooled_inner = pd.DataFrame(pooled_inner_records)
inner_summary = pooled_inner.groupby('model').agg(
    roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'),
    pr_auc_mean=('pr_auc', 'mean'),
    recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'),
    specificity_mean=('specificity', 'mean'),
    f1_mean=('f1', 'mean'),
    brier_mean=('brier', 'mean'),
    false_negatives_mean=('false_negatives', 'mean'),
).reset_index().merge(inner_worst_site, on='model')

print('INNER DEVELOPMENT SUMMARY — used for development comparison only')
display(inner_summary.sort_values(
    ['roc_auc_worst', 'recall_worst'],
    ascending=False,
).round(6))

print('INNER DEVELOPMENT SITE SUMMARY')
display(inner_site_summary.sort_values(['model', 'validation_site']).round(6))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(
    data=inner_site_summary,
    x='validation_site',
    y='roc_auc',
    hue='model',
    ax=axes[0],
)
axes[0].set_title('Inner validation ROC-AUC by development hospital')
axes[0].set_ylim(0.4, 1.0)
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(
    data=inner_site_summary,
    x='validation_site',
    y='recall',
    hue='model',
    ax=axes[1],
)
axes[1].set_title('Inner validation Recall at threshold 0.50')
axes[1].set_ylim(0.0, 1.0)
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'inner_development_metrics_by_site.png',
    dpi=180,
    bbox_inches='tight',
)
plt.show()

## Final fit and fixed Hungarian test

Trước khi chạy cell này, khóa toàn bộ quyết định:

- baseline và hyperparameters giữ nguyên;
- preprocessing giữ nguyên;
- threshold giữ nguyên ở 0.50;
- final seed giữ nguyên ở 42;
- không dùng kết quả Hungarian để chọn model.

Cell này fit mỗi baseline trên toàn bộ Cleveland + Switzerland + VA rồi đánh giá Hungarian một lần. Không có bước tuning hay điều chỉnh nào sau kết quả này.

In [ ]:
# FINAL TEST CELL — execute once after all decisions are locked.
final_test_records = []
final_test_predictions = []

for model_name in MODEL_NAMES:
    pipeline = make_pipeline(model_name, FINAL_SEED)
    started = time.perf_counter()
    pipeline.fit(
        development[FEATURES],
        development[TARGET],
    )
    fit_seconds = time.perf_counter() - started
    probability = pipeline.predict_proba(final_test[FEATURES])[:, 1]

    final_test_records.append(metric_record(
        evaluation='fixed_hungarian_final_test',
        model_name=model_name,
        validation_site=FINAL_TEST_SITE,
        seed=FINAL_SEED,
        y_true=final_test[TARGET].to_numpy(),
        probability=probability,
        fit_seconds=fit_seconds,
    ))

    final_test_predictions.append(pd.DataFrame({
        'row_index_in_hungarian': np.arange(len(final_test)),
        'model': model_name,
        'y_true': final_test[TARGET].to_numpy(),
        'probability': probability,
        'prediction_at_0_50': (probability >= THRESHOLD).astype('int8'),
    }))

final_test_results = pd.DataFrame(final_test_records)
final_test_predictions = pd.concat(
    final_test_predictions,
    ignore_index=True,
)

print('FINAL FIXED HUNGARIAN TEST — one evaluation per baseline model')
display(final_test_results.round(6))

In [ ]:
# Report development ranking separately from the final fixed-test result.
development_ranking = inner_summary.sort_values(
    ['roc_auc_worst', 'recall_worst', 'brier_mean'],
    ascending=[False, False, True],
).reset_index(drop=True)

print('Development ranking is based only on inner CV.')
display(development_ranking.round(6))

print(
    'Interpretation: use inner CV to discuss development robustness; '
    'report Hungarian as the fixed external holdout result. '
    'Do not retune after seeing Hungarian.'
)

inner_cv_results.to_csv(OUTPUT_DIR / 'inner_cv_results.csv', index=False)
inner_site_summary.to_csv(OUTPUT_DIR / 'inner_cv_site_summary.csv', index=False)
inner_summary.to_csv(OUTPUT_DIR / 'inner_cv_summary.csv', index=False)
pooled_inner.to_csv(OUTPUT_DIR / 'inner_cv_pooled_by_seed.csv', index=False)
final_test_results.to_csv(
    OUTPUT_DIR / 'final_hungarian_test_results.csv',
    index=False,
)
final_test_predictions.to_csv(
    OUTPUT_DIR / 'final_hungarian_predictions.csv',
    index=False,
)

run_config = {
    'notebook': '26_UCI_Multicenter_Fixed_Hungarian_Baseline_Colab',
    'dataset': 'UCI Heart Disease, four processed cohorts',
    'expected_rows': EXPECTED_ROWS,
    'features': FEATURES,
    'target_rule': 'target = (num > 0)',
    'final_test_site': FINAL_TEST_SITE,
    'development_sites': DEVELOPMENT_SITES,
    'validation': (
        'Inner 3-fold GroupKFold by hospital; '
        'each fold uses 2 train hospitals + 1 validation hospital'
    ),
    'final_fit': 'Cleveland + Switzerland + VA',
    'final_test_policy': (
        'Hungarian held out until final evaluation; one fixed final seed'
    ),
    'models': MODEL_NAMES,
    'baseline': {
        'preprocessing': (
            'P1; train-split median/mode imputation; missing indicators; '
            'one-hot encoding; scaling'
        ),
        'class_weight': 'balanced',
        'threshold': THRESHOLD,
        'optuna': False,
        'smote': False,
        'synthetic_data': False,
        'stacking': False,
    },
    'inner_seeds': list(INNER_SEEDS),
    'final_seed': FINAL_SEED,
    'exploratory_caveat': (
        'LOCO results from Colab 24/25 were already inspected; '
        'Hungarian is not a blind absolute test.'
    ),
    'data_source': str(LOCAL_DATA_DIR) if LOCAL_DATA_DIR else 'UCI URL fallback',
}
(OUTPUT_DIR / 'run_config.json').write_text(
    json.dumps(run_config, indent=2),
    encoding='utf-8',
)

zip_path = shutil.make_archive(
    '/content/uci_multicenter_fixed_hungarian_baseline_results',
    'zip',
    OUTPUT_DIR,
)
print('Saved artifacts:', OUTPUT_DIR)
print('ZIP:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Colab; ZIP remains at:', zip_path)

## Checklist trước khi báo cáo

- [x] Hungarian cố định làm external holdout.
- [x] Cleveland, Switzerland, VA chỉ dùng cho development.
- [x] Inner CV theo hospital: 2 train + 1 validation.
- [x] Final fit trên cả 3 development hospitals.
- [x] P1 và preprocessing fit trong từng training fold.
- [x] Baseline cũ: Logistic Regression + LightGBM, class-balanced, threshold 0.50.
- [x] Không Optuna, SMOTE, synthetic data hoặc stacking.
- [x] Hungarian chỉ xuất hiện ở cell final test.
- [ ] Không điều chỉnh model sau khi đã xem kết quả Hungarian.
- [ ] Nếu cần một đánh giá mù tuyệt đối, bổ sung hospital/dataset thứ 5.